# MRCD Multi-Round Collaborative Detection Pipeline

Notebook này thực hiện so sánh đánh giá giữa mô hình SLM thuần túy (Baseline) và hệ thống MRCD đa vòng lặp.

### 1. Tải Mã Nguồn từ GitHub
Thay thế link GitHub bên dưới bằng link repository của bạn.

In [ ]:
import os

GITHUB_REPO = "https://github.com/USERNAME/MRCD.git"  # CẬP NHẬT TẠI ĐÂY
REPO_NAME = "MRCD"

if not os.path.exists(REPO_NAME):
    !git clone {GITHUB_REPO}
%cd {REPO_NAME}

### 2. Cài đặt Thư viện

In [ ]:
!pip install -r requirements.txt --quiet

### 3. Cấu hình Môi trường

In [ ]:
import os
import sys
import pandas as pd
from tqdm.auto import tqdm

sys.path.append(os.getcwd())

os.environ["MRCD_DATA_DIR"] = "/kaggle/input/datasets/chinhde/twitter15-16"
os.environ["MRCD_MODEL_PATH"] = "/kaggle/input/models/chinhde/roberta-fine-v2/pytorch/default/1"
os.environ["LLM_MODEL_NAME"] = "meta-llama/Meta-Llama-3-8B-Instruct"
os.environ["MRCD_RESULTS_CSV"] = "/kaggle/working/mrcd_results.csv"

from src.slm.model import IntegratedSLM
from src.pipeline.runner import run_mrcd_pipeline
from src.slm.dataset import load_data_from_csv
from src.utils import set_seed
from src.evaluation.metrics import evaluate_and_plot, compare_models

set_seed(42)
print("Setup completed.")

### 4. Nạp Dữ liệu và Mô hình

In [ ]:
slm = IntegratedSLM()
train_texts, train_labels, test_texts, test_labels = load_data_from_csv()

# Chọn số lượng mẫu để test
SAMPLE_SIZE = 50 
sample_events = test_texts[:SAMPLE_SIZE]
sample_ground_truth = test_labels[:SAMPLE_SIZE]

### 5. Đánh giá Baseline (SLM đơn thuần)
Trước khi chạy MRCD, chúng ta kiểm tra xem mô hình SLM hiện tại dự đoán tập test như thế nào.

In [ ]:
print(f"Evaluating Baseline SLM on {SAMPLE_SIZE} samples...")
y_pred_baseline = []
for text in tqdm(sample_events, desc="Baseline Inference"):
    pred, conf, _ = slm.inference(text)
    y_pred_baseline.append(pred)

baseline_metrics = evaluate_and_plot(
    sample_ground_truth, 
    y_pred_baseline, 
    labels=['Real', 'Fake'], 
    model_name="Baseline SLM"
)

### 6. Chạy MRCD Pipeline (Đa vòng lặp)
Hệ thống sẽ thực hiện truy xuất kiến thức, phối hợp LLM + SLM qua nhiều vòng để lọc nhiễu và cập nhật nhãn.

In [ ]:
print(f"Starting MRCD Pipeline for {SAMPLE_SIZE} events...")
mrcd_output = run_mrcd_pipeline(
    events=sample_events,
    slm=slm,
    max_rounds=3
)

# Thu thập nhãn dự đoán cuối cùng từ output
y_pred_mrcd = [res['label'] for res in mrcd_output['results']]

mrcd_metrics = evaluate_and_plot(
    sample_ground_truth, 
    y_pred_mrcd, 
    labels=['Real', 'Fake'], 
    model_name="MRCD Pipeline"
)

### 7. So sánh đối chiếu Kết quả
Bảng so sánh và biểu đồ metrics giữa Baseline và MRCD.

In [ ]:
comparison_df = compare_models({
    "Baseline SLM": baseline_metrics,
    "MRCD Pipeline": mrcd_metrics
})

display(comparison_df)